<a href="https://colab.research.google.com/github/Lyv-ux/DI_Bootcamp/blob/main/W9D3_DC2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install -U langchain langchain-community langchain-huggingface langchain-groq python-dotenv tavily-python sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.13
    Uninstalling langchain-1.3.13:
      Successfully uninstalled langchain-1.3.13
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour

In [7]:
!pip install -U --force-reinstall langchain langchain-community langchain-huggingface langchain-groq python-dotenv tavily-python sentence-transformers faiss-cpu

  Using cached langchain-1.3.14-py3-none-any.whl.metadata (6.1 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_huggingface-1.2.2-py3-none-any.whl.metadata (4.0 kB)
  Using cached langchain_groq-1.1.3-py3-none-any.whl.metadata (2.9 kB)
  Using cached tavily_python-0.7.26-py3-none-any.whl.metadata (12 kB)
  Using cached faiss_cpu-1.14.3-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (7.8 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 8.3 MB/s eta 0:00:00
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached langchain_classic-1.0.8-py3-none-any.whl.metadata (5.1 kB)
  Using cached pydantic_settings-2.14.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached groq-0.37.1-py3-none-any.whl.metadata (16 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━

In [1]:
# ============================
# AGENTIC RAG NOTEBOOK - ONE CELL
# ============================

# Imports
import os
from dotenv import load_dotenv

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain_community.tools.tavily_search import TavilySearchResults

from langchain_core.tools import Tool
from langchain.agents.react.agent import create_react_agent
from langchain.agents.agent_executor import AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

# Load environment variables
load_dotenv()

# API Keys
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

# ----------------------------
# Load local knowledge base
# ----------------------------
try:
    if os.path.exists("knowledge.txt"):
        loader = TextLoader("knowledge.txt", encoding="utf-8")
        docs = loader.load()

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=50
        )

        documents = splitter.split_documents(docs)

        embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )

        vectordb = FAISS.from_documents(documents, embeddings)

        retriever = vectordb.as_retriever(search_kwargs={"k": 3})

        def local_retrieval(query):
            results = retriever.get_relevant_documents(query)
            return "\n\n".join([doc.page_content for doc in results])

    else:
        def local_retrieval(query):
            return "No local knowledge base found."

except Exception as e:
    def local_retrieval(query):
        return f"Retrieval error: {str(e)}"

# ----------------------------
# LLM (Groq)
# ----------------------------
llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name="llama3-8b-8192",
    temperature=0
)

# ----------------------------
# Tools
# ----------------------------

retriever_tool = Tool(
    name="LocalKnowledgeBase",
    func=local_retrieval,
    description="Use this tool for questions related to local documents."
)

tavily_tool = TavilySearchResults(
    max_results=5
)

tools = [
    retriever_tool,
    tavily_tool
]

# ----------------------------
# Agent
# ----------------------------
# Define the prompt for the agent
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Respond to the user's query as best as you can."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

# Create the agent runnable
agent_runnable = create_react_agent(llm, tools, prompt)

# Create the AgentExecutor
agent = AgentExecutor(
    agent=agent_runnable,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

# ----------------------------
# Main Function for Streamlit
# ----------------------------
def ask_agent(question):
    """
    Function called by app.py
    """
    try:
        response = agent.invoke({"input": question})

        if isinstance(response, dict):
            return response.get("output", str(response))

        return str(response)

    except Exception as e:
        return f"Error: {str(e)}"

# ----------------------------
# Quick Test
# ----------------------------
print(
    ask_agent(
        "What is Retrieval-Augmented Generation (RAG)?"
    )
)

/tmp/ipykernel_3437/4151913832.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


ImportError: cannot import name 'create_react_agent' from 'langchain.agents' (/usr/local/lib/python3.12/dist-packages/langchain/agents/__init__.py)